# POI Enrichment with OpenStreetMap

This notebook creates external point-of-interest features for the outlet potential model. It uses the Overpass API to fetch OpenStreetMap POIs for the target submission outlets, caches the raw response, and computes outlet-level count and nearest-distance features locally.

## 1. Setup

In [1]:
from __future__ import annotations

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from sklearn.neighbors import BallTree

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'Notebooks' else Path.cwd()
DATASETS_DIR = PROJECT_ROOT / 'Datasets'
BRONZE_DIR = PROJECT_ROOT / 'data' / 'bronze'
GOLD_DIR = PROJECT_ROOT / 'data' / 'gold'
RESULTS_DIR = PROJECT_ROOT / 'Results'

for directory in [BRONZE_DIR, GOLD_DIR, RESULTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

OVERPASS_URL = 'https://overpass-api.de/api/interpreter'
RAW_POI_CACHE = BRONZE_DIR / 'poi_overpass_raw.json'
POI_FEATURE_PATH = GOLD_DIR / 'outlet_poi_features.csv'

# If the official template exists, enrich those row_ids. Otherwise use the current
# platform submission file, which is currently the 914-row fallback.
TEMPLATE_CANDIDATES = [
    DATASETS_DIR / 'sample_submission.csv',
    DATASETS_DIR / 'submission_template.csv',
    DATASETS_DIR / 'test.csv',
    RESULTS_DIR / 'smil_labs_predictions.csv',
]

RADIUS_KM = [0.25, 0.5, 1.0, 2.0]
EARTH_RADIUS_KM = 6371.0088

print(PROJECT_ROOT)

d:\projects\Data-Storm-2026


## 2. Select Target Outlets

In [2]:
outlets = pd.read_csv(DATASETS_DIR / 'outlet_master.csv')
coords = pd.read_csv(DATASETS_DIR / 'outlet_coordinates.csv')
coords['Latitude'] = pd.to_numeric(coords['Latitude'], errors='coerce')
coords['Longitude'] = pd.to_numeric(coords['Longitude'], errors='coerce')
coords['valid_coordinates'] = coords['Latitude'].between(5.5, 10.2) & coords['Longitude'].between(79.0, 82.1)

template_path = next((path for path in TEMPLATE_CANDIDATES if path.exists()), None)
if template_path is None:
    raise FileNotFoundError('No submission/template file found for selecting target outlets.')

template = pd.read_csv(template_path)
if 'row_id' in template.columns:
    target_ids = template['row_id'].astype(str).drop_duplicates()
elif 'Outlet_ID' in template.columns:
    target_ids = template['Outlet_ID'].astype(str).drop_duplicates()
else:
    raise ValueError(f'{template_path} must contain row_id or Outlet_ID')

target_outlets = coords.loc[coords['Outlet_ID'].astype(str).isin(set(target_ids)) & coords['valid_coordinates']].copy()
target_outlets = target_outlets[['Outlet_ID', 'Latitude', 'Longitude']].drop_duplicates('Outlet_ID')

print(f'Template source: {template_path.relative_to(PROJECT_ROOT)}')
print(f'Target outlets with valid coordinates: {len(target_outlets):,}')
target_outlets.head()

FileNotFoundError: No submission/template file found for selecting target outlets.

## 3. Build Overpass Query

In [ ]:
POI_FILTERS = {
    'education': [
        ('amenity', 'school'), ('amenity', 'college'), ('amenity', 'university'), ('amenity', 'kindergarten')
    ],
    'transport': [
        ('highway', 'bus_stop'), ('amenity', 'bus_station'), ('railway', 'station'), ('railway', 'halt')
    ],
    'market_retail': [
        ('shop', 'supermarket'), ('shop', 'convenience'), ('shop', 'mall'), ('amenity', 'marketplace')
    ],
    'healthcare': [
        ('amenity', 'hospital'), ('amenity', 'clinic'), ('amenity', 'pharmacy'), ('healthcare', 'hospital'), ('healthcare', 'clinic')
    ],
    'food_service': [
        ('amenity', 'restaurant'), ('amenity', 'cafe'), ('amenity', 'fast_food'), ('shop', 'bakery')
    ],
    'office_finance': [
        ('amenity', 'bank'), ('amenity', 'atm'), ('office', 'company'), ('office', 'government')
    ],
    'religious': [
        ('amenity', 'place_of_worship')
    ],
    'tourism_hotel': [
        ('tourism', 'hotel'), ('tourism', 'guest_house'), ('tourism', 'attraction'), ('tourism', 'museum')
    ],
}

def outlet_bbox(targets: pd.DataFrame, margin_degrees: float = 0.03) -> str:
    south = targets['Latitude'].min() - margin_degrees
    north = targets['Latitude'].max() + margin_degrees
    west = targets['Longitude'].min() - margin_degrees
    east = targets['Longitude'].max() + margin_degrees
    return f'{south},{west},{north},{east}'


def build_category_query(category: str, bbox: str) -> str:
    clauses = []
    for key, value in POI_FILTERS[category]:
        clauses.append(f'node["{key}"="{value}"]({bbox});')
        clauses.append(f'way["{key}"="{value}"]({bbox});')
    body = '\n  '.join(clauses)
    return f'''[out:json][timeout:120];
(
  {body}
);
out center tags;'''

bbox = outlet_bbox(target_outlets)
category_queries = {category: build_category_query(category, bbox) for category in POI_FILTERS}
print(f'Bounding box: {bbox}')
print(category_queries['education'][:1000])



## 4. Fetch or Load Cached POIs

In [ ]:
def fetch_overpass_category(category: str, query: str, cache_dir: Path, use_cache: bool = True) -> dict:
    cache_path = cache_dir / f'poi_overpass_{category}.json'
    if use_cache and cache_path.exists():
        print(f'Loading cached {category}: {cache_path.relative_to(PROJECT_ROOT)}')
        return json.loads(cache_path.read_text(encoding='utf-8'))

    print(f'Fetching {category} POIs from Overpass API...')
    response = requests.post(
        OVERPASS_URL,
        data={'data': query},
        headers={'User-Agent': 'Data-Storm-2026-POI-Enrichment/1.0'},
        timeout=180,
    )
    if response.status_code != 200:
        print(f'Overpass returned {response.status_code} for {category}')
        print(response.text[:1000])
        return {'elements': []}
    payload = response.json()
    cache_path.write_text(json.dumps(payload), encoding='utf-8')
    time.sleep(2)
    return payload


payloads = {}
for category, category_query in category_queries.items():
    payloads[category] = fetch_overpass_category(category, category_query, BRONZE_DIR, use_cache=True)

for category, payload in payloads.items():
    print(category, len(payload.get('elements', [])))



## 5. Parse POIs

In [ ]:
def categorize_poi(tags: dict, fallback_category: str) -> str:
    for category, filters in POI_FILTERS.items():
        for key, value in filters:
            if tags.get(key) == value:
                return category
    return fallback_category


rows = []
for fallback_category, payload in payloads.items():
    for element in payload.get('elements', []):
        tags = element.get('tags', {})
        category = categorize_poi(tags, fallback_category)
        lat = element.get('lat') or element.get('center', {}).get('lat')
        lon = element.get('lon') or element.get('center', {}).get('lon')
        if lat is None or lon is None:
            continue
        rows.append({
            'osm_type': element.get('type'),
            'osm_id': element.get('id'),
            'poi_category': category,
            'Latitude': lat,
            'Longitude': lon,
            'name': tags.get('name'),
        })

poi = pd.DataFrame(rows)
if not poi.empty:
    poi = poi.drop_duplicates(['osm_type', 'osm_id', 'poi_category'])
poi.to_csv(GOLD_DIR / 'poi_cleaned.csv', index=False)
print(poi.shape)
poi['poi_category'].value_counts() if not poi.empty else poi



## 6. Build Outlet-Level POI Features

In [ ]:
def empty_poi_features(outlet_ids: pd.Series) -> pd.DataFrame:
    output = pd.DataFrame({'Outlet_ID': outlet_ids.astype(str).values})
    for category in POI_FILTERS:
        for radius in RADIUS_KM:
            output[f'poi_{category}_count_{str(radius).replace(".", "p")}km'] = 0
        output[f'poi_{category}_nearest_km'] = 5.0
    output['poi_total_count_1km'] = 0
    output['poi_total_count_2km'] = 0
    output['poi_demand_score'] = 0.0
    return output


poi_features = empty_poi_features(target_outlets['Outlet_ID'])

if not poi.empty:
    outlet_rad = np.radians(target_outlets[['Latitude', 'Longitude']].to_numpy())
    poi_features = pd.DataFrame({'Outlet_ID': target_outlets['Outlet_ID'].astype(str).values})

    total_1km = np.zeros(len(target_outlets), dtype=float)
    total_2km = np.zeros(len(target_outlets), dtype=float)

    for category in POI_FILTERS:
        category_poi = poi.loc[poi['poi_category'].eq(category)]
        if category_poi.empty:
            for radius in RADIUS_KM:
                poi_features[f'poi_{category}_count_{str(radius).replace(".", "p")}km'] = 0
            poi_features[f'poi_{category}_nearest_km'] = 5.0
            continue

        poi_rad = np.radians(category_poi[['Latitude', 'Longitude']].to_numpy())
        tree = BallTree(poi_rad, metric='haversine')
        for radius in RADIUS_KM:
            counts = tree.query_radius(outlet_rad, r=radius / EARTH_RADIUS_KM, count_only=True)
            column = f'poi_{category}_count_{str(radius).replace(".", "p")}km'
            poi_features[column] = counts.astype(int)
            if radius == 1.0:
                total_1km += counts
            if radius == 2.0:
                total_2km += counts
        dist, _ = tree.query(outlet_rad, k=1)
        poi_features[f'poi_{category}_nearest_km'] = np.minimum(dist[:, 0] * EARTH_RADIUS_KM, 5.0)

    poi_features['poi_total_count_1km'] = total_1km.astype(int)
    poi_features['poi_total_count_2km'] = total_2km.astype(int)
    density_rank = pd.Series(total_1km + 0.5 * total_2km).rank(pct=True)
    nearest_columns = [column for column in poi_features.columns if column.endswith('_nearest_km')]
    nearest_score = 1 - poi_features[nearest_columns].min(axis=1).rank(pct=True)
    poi_features['poi_demand_score'] = (0.75 * density_rank + 0.25 * nearest_score).fillna(0).clip(0, 1)

poi_features.to_csv(POI_FEATURE_PATH, index=False)
print(f'Wrote {POI_FEATURE_PATH.relative_to(PROJECT_ROOT)}')
print(poi_features.shape)
poi_features.head()